# 03 — Customer Funnel Analysis

This notebook constructs the chronological customer funnel: product view, cart, and purchase.

## Coverage
- Funnel stage definitions
- Event sequencing and chronology
- Unique customer counts
- Step and overall conversion
- Drop-off analysis
- Channel, device, and campaign breakdowns
- Funnel metrics export

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "ecommerce_funnel_events_clean.csv"
if not DATA_PATH.exists():
    DATA_PATH = ROOT / "data" / "ecommerce_funnel_events.csv"
REPORTS_PATH = ROOT / "reports"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Clean or normalized dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True, errors="coerce")
df = df.sort_values(["user_id", "event_timestamp"])

STAGES = ["product_view", "cart", "purchase"]
counts = df.groupby("stage")["user_id"].nunique().reindex(STAGES)

rows = []
for i, stage in enumerate(STAGES):
    users = int(counts[stage])
    previous = users if i == 0 else int(counts[STAGES[i - 1]])
    overall = users / int(counts[STAGES[0]]) if counts[STAGES[0]] else 0.0
    drop_off = 0 if i == 0 else previous - users
    rows.append({
        "stage": stage,
        "users": users,
        "step_conversion_rate": 1.0 if i == 0 else users / previous if previous else 0.0,
        "overall_conversion_rate": overall,
        "drop_off_users": drop_off,
        "drop_off_rate": 0.0 if i == 0 else drop_off / previous if previous else 0.0,
    })

funnel = pd.DataFrame(rows)
display(funnel)

segment_columns = [column for column in ["channel", "device", "campaign"] if column in df.columns]
segment_summary = (
    df.pivot_table(index=segment_columns + ["stage"], values="user_id", aggfunc="nunique", fill_value=0)
    .reset_index()
    .rename(columns={"user_id": "users"})
    if segment_columns
    else pd.DataFrame(columns=["stage", "users"])
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(funnel["stage"], funnel["users"], color=["#4C78A8", "#F2CF5B", "#59A14F"])
ax.invert_yaxis()
ax.set_title("Customer Funnel — Unique Users")
ax.set_xlabel("Unique users")
plt.tight_layout()
plt.savefig(REPORTS_PATH / "funnel_analysis.png", dpi=150)
plt.show()

funnel.to_csv(REPORTS_PATH / "funnel_summary.csv", index=False)
segment_summary.to_csv(REPORTS_PATH / "segment_summary.csv", index=False)
display(segment_summary.head(30))
